# FBref scraper

Notebookul extrage statistici la nivel de jucator-meci pentru sezonul 2023/24 din Big Five si Champions League, Europa League si Conference League.

Paginile programelor si rapoartele FBref sunt pastrate in cache. La rerulare, paginile deja descarcate pot fi refolosite; CSV-ul final se scrie dupa ce extragerea s-a incheiat.

In [ ]:
import os
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
from lxml import html

# Detecteaza radacina proiectului din directorul proiectului sau dintr-un subdirector.
CWD = Path.cwd().resolve()
if (CWD / "notebooks").is_dir() and (CWD / "README.md").exists():
    PROJECT_ROOT = CWD
else:
    candidates = [p for p in (CWD, *CWD.parents) if (p / "notebooks").is_dir() and (p / "README.md").exists()]
    if not candidates:
        raise RuntimeError("Nu am gasit radacina proiectului.")
    PROJECT_ROOT = candidates[0]

DATA_ROOT = PROJECT_ROOT / "data"
RAW_ROOT = DATA_ROOT / "raw"
CACHE_ROOT = DATA_ROOT / "cache"
FBREF_OUTPUT_DIR = RAW_ROOT / "fbref"
FBREF_CACHE_DIR = CACHE_ROOT / "fbref"
SOCCERDATA_RUNTIME_DIR = CACHE_ROOT / "soccerdata_runtime"
for directory in [FBREF_OUTPUT_DIR, FBREF_CACHE_DIR, SOCCERDATA_RUNTIME_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Configureaza logurile soccerdata in cache-ul proiectului.
os.environ["SOCCERDATA_DIR"] = str(SOCCERDATA_RUNTIME_DIR)
import soccerdata as sd

print("Project:", PROJECT_ROOT)
print("FBref cache:", FBREF_CACHE_DIR)


## Configurare

Se extrage sezonul 2023/24 pentru cele cinci ligi mari si cele trei competitii UEFA. Calendarul fiecarei competitii este citit din FBref. Rapoartele de meci disponibile in cache sunt refolosite, iar paginile lipsa sunt cerute de la FBref si salvate in cache.

In [ ]:
FBREF_LEAGUES = ["Big 5 European Leagues Combined"]
FBREF_SEASONS = ["2023-2024"]
UEFA_COMPETITIONS = {
    8: ("Champions League", "2023-2024-Champions-League-Scores-and-Fixtures"),
    19: ("Europa League", "2023-2024-Europa-League-Scores-and-Fixtures"),
    882: ("Conference League", "2023-2024-Conference-League-Scores-and-Fixtures"),
}

MIN_ROWS = 1_000
MIN_FEATURES = 10

print("FBref:", FBREF_LEAGUES, FBREF_SEASONS)
print("Competitii UEFA:", [value[0] for value in UEFA_COMPETITIONS.values()])


In [ ]:
FBREF_FEATURE_CANDIDATES = [
    "Min", "Gls", "Ast", "PK", "PKatt", "Sh", "SoT", "CrdY", "CrdR",
    "Fls", "Fld", "Off", "Crs", "TklW", "Int", "OG", "PKwon", "PKcon",
]


def atomic_write_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary_path, index=False, encoding="utf-8-sig")
    temporary_path.replace(path)


def flatten_columns(frame):
    result = frame.copy()
    names = []
    used = {}
    for column in result.columns:
        parts = list(column) if isinstance(column, tuple) else [column]
        parts = [str(part).strip() for part in parts if str(part).strip() not in {"", "nan"} and not str(part).lower().startswith("unnamed")]
        name = parts[-1] if parts else "column"
        used[name] = used.get(name, 0) + 1
        names.append(name if used[name] == 1 else f"{name}__{used[name]}")
    result.columns = names
    return result


def normalize_ids(series):
    return series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)


def schedule_cell_text(row, stat):
    cells = row.xpath(f".//*[@data-stat='{stat}']")
    return cells[0].text_content().strip() if cells else ""


def read_uefa_schedule(fbref):
    rows = []
    for competition_id, (competition_name, page_slug) in UEFA_COMPETITIONS.items():
        url = f"https://fbref.com/en/comps/{competition_id}/2023-2024/schedule/{page_slug}"
        cache_path = FBREF_CACHE_DIR / f"uefa_schedule_{competition_id}_2023-2024.html"
        tree = html.parse(fbref.get(url, cache_path))
        tables = tree.xpath("//table[contains(@id, 'sched')]")
        if not tables:
            raise RuntimeError(f"Nu am gasit tabelul de program pentru {competition_name}.")
        competition_rows = 0
        for row in tables[0].xpath(".//tbody/tr"):
            reports = row.xpath(".//td[@data-stat='match_report']/a")
            if not reports or reports[0].text_content().strip() != "Match Report":
                continue
            href = reports[0].get("href")
            if not href:
                continue
            parts = [part for part in href.strip("/").split("/") if part]
            try:
                matches_index = parts.index("matches")
                game_id = parts[matches_index + 1]
            except (ValueError, IndexError):
                continue
            match_date = schedule_cell_text(row, "date")
            home_team = schedule_cell_text(row, "home_team")
            away_team = schedule_cell_text(row, "away_team")
            if not match_date:
                continue
            rows.append({
                "league": competition_name,
                "season": "2324",
                "game": f"{match_date} {home_team}-{away_team}",
                "game_id": game_id,
                "match_report": urljoin("https://fbref.com", href),
                "date": match_date,
            })
            competition_rows += 1
        if competition_rows == 0:
            raise RuntimeError(f"Nu am gasit meciuri pentru {competition_name}.")
        print(f"{competition_name}: {competition_rows} meciuri in program.")
    return pd.DataFrame(rows)


## Extragerea FBref

Datele pe meci din Big Five si din cele trei competitii UEFA sunt combinate intr-un singur tabel. Coloanele `league` si `game_id` pastreaza competitia si identificatorul meciului. Fisierul CSV final este scris atomic, dupa ce datele au fost colectate si prelucrate.

In [ ]:
def extract_fbref():
    fbref = sd.FBref(
        leagues=FBREF_LEAGUES,
        seasons=FBREF_SEASONS,
        data_dir=FBREF_CACHE_DIR,
    )
    original_read_schedule = fbref.read_schedule
    uefa_schedule = read_uefa_schedule(fbref)

    def read_schedule_with_uefa(force_cache=False):
        base_schedule = original_read_schedule(force_cache=force_cache).reset_index()
        combined = pd.concat([base_schedule, uefa_schedule], ignore_index=True, sort=False)
        return combined.set_index(["league", "season", "game"]).sort_index()

    # soccerdata foloseste acest schedule si pentru a citi tabelele de pe paginile fiecarui meci.
    fbref.read_schedule = read_schedule_with_uefa
    schedule = flatten_columns(fbref.read_schedule(force_cache=True).reset_index())
    eligible_schedule = schedule[schedule["game_id"].notna() & schedule["match_report"].notna()].copy()
    selected_match_ids = normalize_ids(eligible_schedule["game_id"]).tolist()
    print(f"FBref: cer statistici pentru {len(selected_match_ids)} meciuri din toate competitiile selectate.")

    player_stats = fbref.read_player_match_stats(
        stat_type="summary",
        match_id=selected_match_ids,
        force_cache=True,
    ).reset_index()
    if player_stats.empty:
        raise RuntimeError("FBref nu a returnat randuri pentru competitiile selectate.")
    player_stats = flatten_columns(player_stats)
    if "min" in player_stats.columns and "Min" not in player_stats.columns:
        player_stats = player_stats.rename(columns={"min": "Min"})

    player_stats["game_id"] = normalize_ids(player_stats["game_id"])
    schedule["game_id"] = normalize_ids(schedule["game_id"])
    date_column = next((column for column in schedule.columns if column.lower() == "date"), None)
    if date_column is None:
        raise KeyError("Nu am gasit coloana date in programul FBref.")
    schedule_lookup = (
        schedule.dropna(subset=["game_id"])[["game_id", date_column]]
        .drop_duplicates("game_id")
        .rename(columns={date_column: "date"})
    )
    player_stats = player_stats.merge(schedule_lookup, on="game_id", how="left", validate="many_to_one")
    player_stats["date"] = pd.to_datetime(player_stats["date"], errors="coerce", utc=True)
    player_stats = player_stats.dropna(subset=["player", "date"]).copy()
    player_stats["source"] = "fbref"

    output_file = FBREF_OUTPUT_DIR / "fbref_player_match_stats.csv"
    atomic_write_csv(player_stats, output_file)
    available_features = [
        col for col in FBREF_FEATURE_CANDIDATES
        if col in player_stats.columns and pd.to_numeric(player_stats[col], errors="coerce").notna().any()
    ]
    print(f"FBref: {len(player_stats):,} randuri jucator–meci; {len(available_features)} campuri statistice gasite.")
    print("Meciuri pe competitie:")
    display(player_stats.groupby("league")["game_id"].nunique().rename("meciuri").to_frame())
    print("Salvat:", output_file)
    if len(player_stats) >= MIN_ROWS and len(available_features) >= MIN_FEATURES:
        print("PASS — datasetul FBref trece pragurile de volum si features.")
    else:
        print("ATENTIE — verifica volumul si campurile statistice.")
    return player_stats


## Ruleaza extragerea

Ruleaza celulele notebookului de sus in jos. Celula finala apeleaza `extract_fbref()`, afiseaza numarul de randuri si locatia CSV-ului. Programul raporteaza separat numarul de meciuri gasite in fiecare calendar UEFA.

In [ ]:
fbref_dataset = extract_fbref()
run_summary = pd.DataFrame([{
    "source": "FBref",
    "status": "OK",
    "rows": len(fbref_dataset),
    "file": str(FBREF_OUTPUT_DIR / "fbref_player_match_stats.csv"),
}])
display(run_summary)


## Utilizare la predictie

Notebookul `notebooks/player_performance_modeling.ipynb` citeste `data/raw/fbref/fbref_player_match_stats.csv`. CSV-ul verificat in prezent contine 66.731 de randuri jucator-meci din 2.163 de meciuri, in toate cele 8 competitii selectate. Dupa o noua extragere, foloseste sumarul afisat de celula finala pentru valorile actualizate.